# Task map

Every policy's success on each of the 40 LIBERO tasks, tasks sorted from easiest to hardest (mean over policies): if the policies fail on the same tasks, the rows look alike.

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt

import utils as u

data_path = Path("../data")
figs_path = Path("figs")

In [ ]:
u.load_data(data_path)
figs_path.mkdir(exist_ok=True)

Failure overlap between the kNN and each VLA, and the per-task correlation (quoted in `numbers.tex`).

In [ ]:
for bench in ("libero", "libero_plus"):
    print(bench, u.failure_overlap(bench))
    print(bench, u.task_corr(bench))

In [ ]:
panels = [(bench, *u.task_matrix(bench)) for bench in ("libero", "libero_plus")]
n = len(u.POLICIES)
cmap = mpl.colors.LinearSegmentedColormap.from_list("succ", ["#ffffff", "#c9dcf4", "#5a9be3", "#104281"])

In [ ]:
fig, axes = plt.subplots(len(panels), 1, figsize=(u.W, 0.6 + 0.18 * n * len(panels) + 0.4 * len(panels)),
                         layout="constrained", squeeze=False)  # fmt: skip
axes = axes[:, 0]
for ax, (bench, rows, m) in zip(axes, panels, strict=True):
    ax.imshow(m, cmap=cmap, vmin=0, vmax=1, aspect="auto", interpolation="nearest")
    ax.set_yticks(range(len(rows)), rows, fontsize=u.FS_SMALL - 2.5)
    for tick, lab in zip(ax.get_yticklabels(), rows, strict=True):
        if lab.startswith("kNN"):
            tick.set_color(u.KNN)
            tick.set_fontweight("bold")
    ax.set_xticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.tick_params(length=0)
    ax.set_title("LIBERO" if bench == "libero" else "LIBERO-plus (perturbed)", fontsize=u.FS_SMALL, color=u.INK2,
                 loc="left")  # fmt: skip
axes[-1].set_xlabel("40 tasks, from easiest to hardest (mean over policies)", fontsize=u.FS_SMALL - 1.5, color=u.INK2)
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=mpl.colors.Normalize(0, 100))
cb = fig.colorbar(sm, ax=axes, location="right", shrink=0.6, pad=0.01, aspect=25)
cb.set_label("task success (%)", fontsize=u.FS_SMALL - 2)
cb.ax.tick_params(labelsize=u.FS_SMALL - 3)
cb.outline.set_visible(False)
fig.savefig(figs_path / "task_map.pdf")